# M5 Forecasting – Walmart daily sales with LightGBM

The goal is to forecast the daily sales of 30,490 Walmart products (3 states, 10 stores) for the next 28 days.

My plan:
1. Load the data and reshape it (one row = one product on one day)
2. Add calendar and price information
3. Build features from the past (lags, rolling means), without any data leakage
4. Check the model on two different periods (backtest)
5. Train LightGBM and compare it with simple baselines
6. Look at what the model learned and where it struggles
7. Build the submission file

## 1. Loading and reshaping the data

In [ ]:
import glob, os, gc
import numpy as np
import pandas as pd

BASE = os.path.dirname(glob.glob('/kaggle/input/**/calendar.csv', recursive=True)[0]) + '/'
print('BASE =', BASE)

FIRST_DAY = 900   # days before this are never loaded
LAST_DAY = 1941
ID_COLS = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

day_cols = [f'd_{i}' for i in range(FIRST_DAY, LAST_DAY + 1)]

sales = pd.read_csv(
    BASE + 'sales_train_evaluation.csv',
    usecols=ID_COLS + day_cols,
    dtype={c: 'int16' for c in day_cols},
)
for c in ID_COLS:
    sales[c] = sales[c].astype('category')

sales = sales.rename(columns={c: int(c[2:]) for c in day_cols})
long = sales.melt(
    id_vars=ID_COLS,
    value_vars=[int(c[2:]) for c in day_cols],
    var_name='d', value_name='sales',
)
long['d'] = long['d'].astype('int16')

del sales; gc.collect()
print(f'{len(long):,} rows | {long.memory_usage(deep=True).sum()/1e9:.2f} GB')

In the original file, each row is a product and each column is a day (1,941 columns). This format doesn't work for time series, so I used `melt` to get one row per product per day.

I only loaded days 900 to 1941. The older data (2011–2013) takes a lot of memory and doesn't help much to forecast 2016. I also used small types (`int16`, `category`) to keep memory low.

Result: **31.8 million rows for only 0.38 GB.**

## 2. Calendar and prices

In [ ]:
calendar = pd.read_csv(BASE + 'calendar.csv')

cal = calendar[['d', 'wm_yr_wk', 'wday', 'month', 'year',
                'event_name_1', 'snap_CA', 'snap_TX', 'snap_WI']].copy()
cal['d'] = cal['d'].str.slice(2).astype('int16')
cal['week'] = pd.to_datetime(calendar['date']).dt.isocalendar().week.astype('int8')
cal['event_name_1'] = cal['event_name_1'].fillna('none').astype('category')
for c in ['wday', 'month', 'snap_CA', 'snap_TX', 'snap_WI']:
    cal[c] = cal[c].astype('int8')
cal['year'] = cal['year'].astype('int16')
cal['wm_yr_wk'] = cal['wm_yr_wk'].astype('int32')

long = long.merge(cal, on='d', how='left')

prices = pd.read_csv(BASE + 'sell_prices.csv')
prices['store_id'] = prices['store_id'].astype(long['store_id'].dtype)
prices['item_id'] = prices['item_id'].astype(long['item_id'].dtype)
prices['wm_yr_wk'] = prices['wm_yr_wk'].astype('int32')
prices['sell_price'] = prices['sell_price'].astype('float32')

long = long.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
del prices; gc.collect()

# remove the periods when the product was not on sale
before = len(long)
long = long[long['sell_price'].notna()].reset_index(drop=True)
print(f'{before:,} -> {len(long):,} rows')

# one SNAP column instead of three
long['snap'] = np.select(
    [long['state_id'] == 'CA', long['state_id'] == 'TX', long['state_id'] == 'WI'],
    [long['snap_CA'], long['snap_TX'], long['snap_WI']],
    default=0).astype('int8')
long['is_event'] = (long['event_name_1'] != 'none').astype('int8')
long.drop(columns=['snap_CA', 'snap_TX', 'snap_WI', 'wm_yr_wk'], inplace=True)

gc.collect()
print(f'{len(long):,} rows x {long.shape[1]} columns | '
      f'{long.memory_usage(deep=True).sum()/1e9:.2f} GB')

For each row I added what we know about the day: day of the week, month, week of the year, events (holidays), SNAP days (food assistance payment days) and the price of the product that week.

There were three SNAP columns (one per state). A store in California doesn't care about the Texas calendar, so I kept only the SNAP value of the store's own state.

Then I removed the rows with no price. No price means the product was not in the store yet. The zeros on these days don't mean "nobody bought it", they mean "there was nothing to buy", and they would teach the model something wrong.

Result: from 31.8 to **30.0 million rows**, 16 columns.

## 3. Features from the past

In [ ]:
g = long.groupby('id', observed=True)['sales']
for lag in [28, 35, 42]:
    long[f'lag_{lag}'] = g.shift(lag).astype('float32')
gc.collect()

gl = long.groupby('id', observed=True)['lag_28']
for w in [7, 28, 60]:
    long[f'rmean_{w}'] = (gl.rolling(w).mean()
                            .reset_index(level=0, drop=True).astype('float32'))
    gc.collect()
long['rstd_28'] = (gl.rolling(28).std()
                     .reset_index(level=0, drop=True).astype('float32'))

long['price_change'] = (long['sell_price'] /
    long.groupby('id', observed=True)['sell_price'].shift(1)).astype('float32')
im = long.groupby('item_id', observed=True)['sell_price'].transform('mean')
long['price_rel'] = (long['sell_price'] / im).astype('float32')
del im, g, gl; gc.collect()

print(f'{len(long):,} rows x {long.shape[1]} columns | '
      f'{long.memory_usage(deep=True).sum()/1e9:.2f} GB')
long[['d', 'sales', 'lag_28', 'rmean_7', 'rmean_28']].tail(10)

The best way to guess how much a product will sell tomorrow is to look at how much it sold before. So I added:

- **Lags**: sales 28, 35 and 42 days ago
- **Rolling means**: average of the last 7, 28 and 60 days
- **Volatility**: how much the sales move (rolling std)
- **Price**: did the price change, and is the product cheap or expensive compared to its usual price?

The important point: the closest lag is 28 days. We forecast 28 days ahead, so at prediction time the last 28 days don't exist yet. Using a shorter lag would be data leakage. For the same reason, the rolling means are computed on `lag_28`, not on the raw sales.

## 4. Train / validation split

In [ ]:
FEATURES = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id',
            'wday', 'month', 'year', 'week', 'event_name_1', 'is_event', 'snap',
            'sell_price', 'price_change', 'price_rel',
            'lag_28', 'lag_35', 'lag_42',
            'rmean_7', 'rmean_28', 'rmean_60', 'rstd_28']
CATS = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'event_name_1']

train = long[(long['d'] >= 1000) & (long['d'] <= 1913)]
valid = long[ long['d'] > 1913]

print(f'train: {len(train):,} rows')
print(f'valid: {len(valid):,} rows')

I split the data **by date**, not randomly: days 1000–1913 for training, and the last 28 days (1914–1941) for validation. With a random split, the model would see future days while predicting past days, which is a form of cheating.

I didn't use `id` as a feature. It has 30,490 different values and the model would just memorize each series. `item_id` + `store_id` carry the same information in a way that generalizes better.

Validation: 853,720 rows = 30,490 products x 28 days, so no series is missing.

## 5. Backtest: checking the model on an earlier period

In [ ]:
import lightgbm as lgb

params = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.1,
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 96,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'verbosity': -1,
}

def rmse(y, yhat):
    return np.sqrt(np.mean((np.asarray(y, dtype='float64') - np.asarray(yhat, dtype='float64')) ** 2))

# same setup, but everything moved 28 days earlier
train_bt = long[(long['d'] >= 1000) & (long['d'] <= 1885)]
valid_bt = long[(long['d'] > 1885) & (long['d'] <= 1913)]
print(f'backtest train: {len(train_bt):,} rows | backtest valid: {len(valid_bt):,} rows')

dtr = lgb.Dataset(train_bt[FEATURES], label=train_bt['sales'], categorical_feature=CATS)
dva = lgb.Dataset(valid_bt[FEATURES], label=valid_bt['sales'],
                  categorical_feature=CATS, reference=dtr)
model_bt = lgb.train(params, dtr, num_boost_round=400, valid_sets=[dva],
                     callbacks=[lgb.early_stopping(30), lgb.log_evaluation(100)])

pred_bt = model_bt.predict(valid_bt[FEATURES], num_iteration=model_bt.best_iteration).clip(0)
bt_base = rmse(valid_bt['sales'], valid_bt['rmean_28'].fillna(0))
bt_lgb = rmse(valid_bt['sales'], pred_bt)
print(f'\nBaseline (28d mean) : {bt_base:.4f}')
print(f'LightGBM            : {bt_lgb:.4f}')
print(f'Improvement vs baseline: {(1 - bt_lgb/bt_base):+.1%}')

del train_bt, valid_bt, dtr, dva, model_bt, pred_bt; gc.collect()

One validation period can be lucky. To check that the gain is real, I repeated the exact same process 28 days earlier: training on days 1000–1885 and testing on days 1886–1913.

If LightGBM also beats the 28-day average on this period, I can trust that the improvement doesn't depend on one particular month. The results of both periods are compared in section 7.

## 6. LightGBM model

In [ ]:
# keep the last 150 days for the submission, free the rest
hist = long[long['d'] > 1941 - 150].copy()
del long; gc.collect()

dtrain = lgb.Dataset(train[FEATURES], label=train['sales'],
                     categorical_feature=CATS)
dvalid = lgb.Dataset(valid[FEATURES], label=valid['sales'],
                     categorical_feature=CATS, reference=dtrain)

model = lgb.train(
    params, dtrain,
    num_boost_round=400,
    valid_sets=[dvalid],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(25)],
)

print(f'\nbest iteration : {model.best_iteration}')
print(f'valid RMSE     : {model.best_score["valid_0"]["rmse"]:.4f}')

**Why LightGBM?** It is fast on 26 million rows, it handles categorical variables directly, and it deals with missing values (the first days have empty lags) on its own.

**Why Tweedie?** Most daily sales are zero. A normal squared-error objective works badly with this kind of data and can predict negative values. Tweedie is made for count data that is "often zero, sometimes positive".

**Early stopping** stops the training when the validation score doesn't improve for 30 rounds. Here it stopped at round 193, with a validation RMSE of **2.152**.

## 7. Comparison with simple baselines

In [ ]:
valid = valid.copy()
valid['pred'] = model.predict(valid[FEATURES],
                              num_iteration=model.best_iteration).clip(0)

z = rmse(valid['sales'], np.zeros(len(valid)))
b = rmse(valid['sales'], valid['rmean_28'].fillna(0))
m = rmse(valid['sales'], valid['pred'])

print(f'Always zero         : {z:.4f}')
print(f'Baseline (28d mean) : {b:.4f}')
print(f'LightGBM            : {m:.4f}')
print(f'\nImprovement vs baseline: {(1 - m/b):+.1%}')

print('\nBoth periods:')
print(pd.DataFrame({
    'period': ['days 1886-1913 (backtest)', 'days 1914-1941 (validation)'],
    'baseline_rmse': [bt_base, b],
    'lightgbm_rmse': [bt_lgb, m],
    'improvement': [f'{1 - bt_lgb/bt_base:+.1%}', f'{1 - m/b:+.1%}'],
}).to_string(index=False))

A score alone doesn't mean much, so I compared the model with two simple rules:

| Method | RMSE |
|---|---|
| Always predict 0 | 3.916 |
| Average of the last 28 days | 2.323 |
| **LightGBM** | **2.152** |

LightGBM reduces the error by **7.4%** compared to the 28-day average. The last table shows the same comparison on the backtest period: if the gain is similar on both periods, the improvement is stable and not a lucky result.

It also shows that a simple average is already a strong baseline for this kind of data.

## 8. What did the model learn?

In [ ]:
import matplotlib.pyplot as plt

imp = pd.DataFrame({'feature': model.feature_name(),
                    'gain': model.feature_importance('gain')}
                  ).sort_values('gain', ascending=False)

imp.head(20).plot(kind='barh', x='feature', y='gain', figsize=(9, 7), legend=False)
plt.gca().invert_yaxis(); plt.title('Feature importance (gain)')
plt.tight_layout(); plt.show()

print(imp.to_string(index=False))

The most important features are, in order:

1. `rmean_60` – the average of the last 60 days (by far)
2. `rmean_28` – the average of the last 28 days
3. `item_id` – which product it is
4. `rstd_28` – how volatile the sales are
5. `rmean_7`, then `wday` (day of the week)

So the model first looks at "how much does this product usually sell", then adjusts with the day of the week. Price and events have a much smaller effect.

For me, the main lesson is that the features I prepared matter more than the choice of model.

## 9. Forecast vs reality for the best-selling products

In [ ]:
top_ids = valid.groupby('id', observed=True)['sales'].sum().nlargest(3).index

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, sid in zip(axes, top_ids):
    s = valid[valid['id'] == sid].sort_values('d')
    ax.plot(s['d'], s['sales'], marker='o', label='actual')
    ax.plot(s['d'], s['pred'], marker='x', label='forecast')
    ax.set_title(str(sid)); ax.legend()
plt.tight_layout(); plt.show()

For the 3 best-selling products, the model follows the general pattern well: higher sales at the weekend, lower during the week.

But it stays too close to the average on the extreme days. When real sales jump to 195, the forecast stays around 180, and on low days the forecast is too high. This is expected: models that minimize squared error prefer to stay near the mean, because a big miss costs a lot.

## 10. Forecasting the next 28 days

In [ ]:
LAST_DAY, H = 1941, 28
BASE_COLS = ['id','item_id','dept_id','cat_id','store_id','state_id','d','sales']

ids = hist.drop_duplicates('id')[['id','item_id','dept_id','cat_id','store_id','state_id']]
days = pd.DataFrame({'d': np.arange(LAST_DAY + 1, LAST_DAY + H + 1, dtype='int16')})
skel = ids.merge(days, how='cross')
skel['sales'] = np.nan

full = pd.concat([hist[BASE_COLS], skel], ignore_index=True)
print(f'{len(full):,} rows ({len(skel):,} in the future)')

# features from the past
g = full.groupby('id', observed=True)['sales']
for lag in [28, 35, 42]:
    full[f'lag_{lag}'] = g.shift(lag).astype('float32')
gl = full.groupby('id', observed=True)['lag_28']
for w in [7, 28, 60]:
    full[f'rmean_{w}'] = (gl.rolling(w).mean()
                            .reset_index(level=0, drop=True).astype('float32'))
full['rstd_28'] = (gl.rolling(28).std()
                     .reset_index(level=0, drop=True).astype('float32'))

# calendar + prices
full = full.merge(cal, on='d', how='left')

prices = pd.read_csv(BASE + 'sell_prices.csv')
prices['store_id'] = prices['store_id'].astype(full['store_id'].dtype)
prices['item_id'] = prices['item_id'].astype(full['item_id'].dtype)
prices['wm_yr_wk'] = prices['wm_yr_wk'].astype('int32')
prices['sell_price'] = prices['sell_price'].astype('float32')
full = full.merge(prices, on=['store_id','item_id','wm_yr_wk'], how='left')
del prices; gc.collect()

full['price_change'] = (full['sell_price'] /
    full.groupby('id', observed=True)['sell_price'].shift(1)).astype('float32')
im = full.groupby('item_id', observed=True)['sell_price'].transform('mean')
full['price_rel'] = (full['sell_price'] / im).astype('float32')
full['snap'] = np.select(
    [full['state_id']=='CA', full['state_id']=='TX', full['state_id']=='WI'],
    [full['snap_CA'], full['snap_TX'], full['snap_WI']],
    default=0).astype('int8')
full['is_event'] = (full['event_name_1'] != 'none').astype('int8')
del im; gc.collect()

fut = full[full['d'] > LAST_DAY].copy()
fut['pred'] = model.predict(fut[FEATURES],
                            num_iteration=model.best_iteration).clip(0)
print(f'{len(fut):,} forecasts (expected: {30490*28:,})')
print(f'mean forecast: {fut["pred"].mean():.3f}')

The days to forecast (1942–1969) are not in the data, so I created them: 28 empty rows per product, added under the last 150 days of history (enough to compute the lags and rolling means). Then I computed the same features and asked the model for predictions.

This is where the "lag ≥ 28" rule pays off: the `lag_28` of the last future day is day 1941, which we already know. So all 28 days can be forecast in one go, without feeding predictions back into the model.

Result: 853,720 forecasts (30,490 x 28).

## 11. Submission file

In [ ]:
F = [f'F{i}' for i in range(1, 29)]

# _evaluation rows -> future forecasts (days 1942-1969)
ev = fut[['id','d','pred']].copy()
ev['id'] = ev['id'].astype(str)
ev['F'] = 'F' + (ev['d'] - LAST_DAY).astype(int).astype(str)
ev = ev.pivot(index='id', columns='F', values='pred').reset_index()[['id'] + F]

# _validation rows -> validation forecasts (days 1914-1941)
va = valid[['id','d','pred']].copy()
va['id'] = va['id'].astype(str).str.replace('_evaluation', '_validation', regex=False)
va['F'] = 'F' + (va['d'] - 1913).astype(int).astype(str)
va = va.pivot(index='id', columns='F', values='pred').reset_index()[['id'] + F]

sub = pd.read_csv(BASE + 'sample_submission.csv')[['id']]
sub = sub.merge(pd.concat([ev, va], ignore_index=True), on='id', how='left').fillna(0)

sub.to_csv('submission.csv', index=False)

print(sub.shape)   # should be (60980, 29)
print('missing values:', sub[F].isna().sum().sum())   # should be 0
print('min / max     :', sub[F].values.min(), sub[F].values.max())
sub.head()

Quick checks before submitting: the shape is (60980, 29), there are no missing values, no negative forecasts (min 0.006), and the max (178.8) is realistic for best-selling products.

## Conclusion

- On 30 million rows, LightGBM reduces the error by **7.4%** compared to a 28-day average.
- The biggest gains came from the features (rolling means of the past), more than from the model itself.
- Being strict with time (split by date, lags ≥ 28 days, a backtest on a second period) is what makes the result trustworthy.

## Limits and next steps

- I tested on two periods of 28 days. More periods (for example around Christmas) would make the check even stronger.
- I used RMSE to compare models. The official M5 metric is WRMSSE, which gives more weight to products with high sales.
- The model underestimates peaks. Next ideas: add features on events (days before/after holidays), or train one model per store.